# Free AI video-to-video test: Wan2.1 VACE 1.3B, depth-controlled (Kaggle or Colab, free GPU)

This notebook keeps the motion and the camera of our Blender clip and lets an open video model repaint it.
It follows the depth of every frame of `beat06-mothers-10s.mp4` and a text prompt for the look.

- Everything here is free:
  - Kaggle's or Colab's free GPU, in your own account;
  - the open weights of **Wan2.1-VACE-1.3B** (Apache-2.0);
  - **Depth-Anything-V2-Small** (Apache-2.0) for the depth maps.
- Your clip is uploaded to Kaggle (or Google, for Colab) under your account. Nothing else leaves the notebook.
- **Untested:** the PC that wrote this has no NVIDIA GPU, so the notebook has not been run yet. Expect to fix
  a line or two the first time (library versions move).
- **Limits:**
  - The 1.3B model is made for 480p (832x480).
  - One call makes 81 frames, about 5 s at 16 fps, so the 10 s clip goes in two pieces.
  - The model card asks for frame counts of 4k+1.
  - Colab's FAQ warns that free runtimes doing "content generation" are commonly terminated, so prefer
    Kaggle.

**Steps on Kaggle:**
1. kaggle.com → New Notebook → File → Import notebook → this file.
2. Add data → Upload → the clip `beat06-mothers-10s.mp4`. It becomes `/kaggle/input/<your-dataset>/...`; set
   `INPUT` below.
3. Settings → Accelerator → GPU T4. Settings → Internet → On (to download the models).
4. Run all. The result is written to `/kaggle/working/wan_vace_out.mp4`; download it from the Output tab.

In [ ]:
!pip -q install "diffusers>=0.34" transformers accelerate imageio imageio-ffmpeg ftfy av

In [ ]:
import glob, math, os
import numpy as np
import torch
import imageio.v3 as iio
from PIL import Image

INPUT = (glob.glob("/kaggle/input/**/beat06-mothers-10s.mp4", recursive=True) or ["beat06-mothers-10s.mp4"])[0]
OUT = "/kaggle/working/wan_vace_out.mp4" if os.path.isdir("/kaggle/working") else "wan_vace_out.mp4"
W, H = 832, 480          # the model's native 480p
FPS = 16                 # Wan generates at 16 fps
CHUNK = 49               # frames per call (4k+1); 81 runs out of memory on a 16 GB T4 at VAE encode
STEPS = 30
SEED = 1846
PROMPT = ("Vienna 1846, dawn. Three poor young women in long plain period dresses, white linen aprons and "
          "bonnets stand before the heavy oak door of a hospital. The woman in the middle clasps her hands "
          "and pleads, then shakes her head; the others look at her, worried. Photorealistic film still, "
          "35 mm lens, natural skin with pores, soft window light, muted colours, shallow depth of field.")
NEGATIVE = ("cartoon, CGI, 3D render, plastic skin, doll, porcelain, oversaturated, blurry, deformed hands, "
            "extra fingers, text, watermark, subtitles, worst quality, low quality")
print(INPUT, "->", OUT)

In [ ]:
# the clip's frames, resampled to 16 fps and scaled to 832x480
meta = iio.immeta(INPUT, plugin="pyav")
src_fps = meta.get("fps", 30)
frames = [Image.fromarray(f).convert("RGB") for f in iio.imiter(INPUT, plugin="pyav")]
step = src_fps / FPS
frames = [frames[min(len(frames) - 1, round(i * step))] for i in range(int(len(frames) / step))]
frames = [f.resize((W, H), Image.LANCZOS) for f in frames]
print(len(frames), "frames at", FPS, "fps")

In [ ]:
# depth for every frame (Depth-Anything-V2-Small, Apache-2.0): the control that keeps shapes and motion
from transformers import pipeline as hf_pipeline
depth_pipe = hf_pipeline(task="depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf",
                         device=0 if torch.cuda.is_available() else -1)
depth = []
for f in frames:
    d = np.array(depth_pipe(f)["depth"]).astype(np.float32)
    d = (255 * (d - d.min()) / max(1e-6, d.max() - d.min())).astype(np.uint8)
    depth.append(Image.fromarray(d).convert("RGB").resize((W, H)))
del depth_pipe
torch.cuda.empty_cache()

In [ ]:
from diffusers import AutoencoderKLWan, WanVACEPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video

model_id = "Wan-AI/Wan2.1-VACE-1.3B-diffusers"
dtype = torch.float16   # not bf16: a T4 reports bf16 as supported, but its memory-efficient attention runs only in fp16
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanVACEPipeline.from_pretrained(model_id, vae=vae, torch_dtype=dtype)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)   # 3.0 for 480p
pipe.enable_model_cpu_offload()      # fits a 16 GB T4
pipe.vae.enable_tiling()             # the VAE encode of the control video is a memory peak
pipe.transformer.set_attention_backend("_native_efficient")   # a T4 has no flash attention: the default falls back to the math kernel and asks for 18 GB

In [ ]:
# the text encoder (umt5-xxl, ~11 GB in fp16) stays on the GPU under model offload and leaves no room
# for the VAE: encode the prompt once, move it to the CPU and hand the pipeline the embeddings
PE, NE = torch.no_grad()(pipe.encode_prompt)(prompt=PROMPT, negative_prompt=NEGATIVE,
                                             do_classifier_free_guidance=True, device=torch.device("cuda"))
pipe.text_encoder.to("cpu")
torch.cuda.empty_cache()
pipe.encode_prompt = lambda *a, **k: (PE, NE)

# pieces of 49 frames; each starts on the last frame of the one before, so the joins can be hidden
out, start = [], 0
while start < len(depth) - 1:
    piece = depth[start:start + CHUNK]
    n = 4 * ((len(piece) - 1) // 4) + 1          # 4k+1 frames
    piece = piece[:n]
    mask = [Image.new("L", (W, H), 255)] * n      # white: generate the whole picture
    result = pipe(video=piece, mask=mask, prompt=PROMPT, negative_prompt=NEGATIVE,
                  height=H, width=W, num_frames=n, num_inference_steps=STEPS, guidance_scale=5.0,
                  generator=torch.Generator("cpu").manual_seed(SEED)).frames[0]
    out.extend(result if not out else result[1:])
    start += n - 1
export_to_video(out, OUT, fps=FPS)
print("wrote", OUT, len(out), "frames")

Optional, on your PC afterwards (free): bring it back to 30 fps and 720p with ffmpeg:

`ffmpeg -i wan_vace_out.mp4 -vf "minterpolate=fps=30,scale=1280:720:flags=lanczos" wan_vace_30fps.mp4`

**What to judge:**
- Is the motion kept (the clasped hands, the head shake)?
- Do the three women stay the same people across the 10 s and across the two pieces?
- Does it read as real?